# Course-End Project: Agentic RAG - Router-Retriever System with PDF and Web Search Tools

**Objective:** Build a multi-agent system using CrewAI where a **Router Agent** classifies an incoming question, and a **Retriever Agent** executes the appropriate retrieval path - either a PDF vector search or a live web search - and returns a grounded, source-cited answer.

**Before running:** Drop 1+ domain-specific PDF(s) into the `pdfs/` folder next to this notebook, and make sure `OPENAI_API_KEY` and `TAVILY_API_KEY` are set in your environment.

## Step 0: Setup and Environment Checks

Confirms both API keys are present and lists the PDFs available for the PDF search tool.

In [1]:
import os
import glob

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set in this environment."
print("OpenAI API key detected.")

assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY is not set. Get one at https://app.tavily.com and set it before continuing."
print("Tavily API key detected.")

PDF_FOLDER = "pdfs"
pdf_paths = sorted(glob.glob(os.path.join(PDF_FOLDER, "*.pdf")))
print(f"Found {len(pdf_paths)} PDF(s): {pdf_paths}")

OpenAI API key detected.
Tavily API key detected.
Found 1 PDF(s): ['pdfs\\trasformer_research_paper-dataset.pdf']


## Step 1: Implement the PDF Search Tool

Uses `crewai_tools.PDFSearchTool`, which performs a RAG-style vector search restricted to the given PDF(s).

In [2]:
from crewai_tools import PDFSearchTool

# One tool per PDF keeps citations unambiguous; combine into a list if you add more PDFs.
pdf_search_tool = PDFSearchTool(pdf=pdf_paths[0]) if pdf_paths else None
if pdf_search_tool is None:
    print("No PDFs found in pdfs/ - add at least one before running the retrieval steps.")
else:
    print(f"PDF search tool ready for: {os.path.basename(pdf_paths[0])}")

c:\claw-code\.venv\Lib\site-packages\fastapi\applications.py:18: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  from fastapi.exception_handlers import (
c:\claw-code\.venv\Lib\site-packages\fastapi\applications.py:30: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  from fastapi.openapi.utils import get_openapi


PDF search tool ready for: trasformer_research_paper-dataset.pdf


## Step 2: Implement the Web Search Tool

Uses LangChain's `TavilySearchResults`, which CrewAI agents can use directly alongside native `crewai_tools`.

In [8]:
from crewai_tools import TavilySearchTool

# Native crewai_tools tool - no LangChain adapter needed, so it's directly Agent-compatible.
web_search_tool = TavilySearchTool(max_results=3)
print("Web search tool ready (Tavily).")

Web search tool ready (Tavily).


## Step 3: Define Agent Roles

- **Router Agent**: analyzes the incoming question and decides whether it needs the static PDF knowledge base or fresh information from the web.
- **Retriever Agent**: has access to both tools, executes the retrieval path chosen by the router, and formulates a grounded, cited answer.

In [10]:
from crewai import Agent

router_agent = Agent(
    role="Query Router",
    goal=(
        "Classify the user's question as either 'pdf' (answerable from the uploaded "
        "domain-specific PDF) or 'web' (needs fresh/current information from the internet). "
        "Respond with a short routing decision and a one-line justification."
    ),
    backstory=(
        "You are an expert dispatcher who has read the uploaded PDF's table of contents "
        "and knows its scope. You never guess - you route based on whether the topic is "
        "covered by the static document or requires real-time/external knowledge."
    ),
    verbose=True,
    allow_delegation=False,
)

retriever_agent = Agent(
    role="Retrieval Specialist",
    goal=(
        "Given the router's decision and the original question, execute the correct retrieval tool (PDF search or web search) and produce a concise, source-grounded answer."
    ),
    backstory=(
        "You are a meticulous research assistant. You only answer using what your tools actually return, and you always cite whether the answer came from the PDF or the web and include the external sources."
    ),
    tools=[tool for tool in [pdf_search_tool, web_search_tool] if tool is not None],
    verbose=True,
    allow_delegation=False,
)

print("Router and Retriever agents defined.")

Router and Retriever agents defined.


## Step 4: Define Tasks (Coordinated Workflow)

The routing task's output feeds into the retrieval task as context, giving a clear, traceable input-output flow between agents.

In [11]:
from crewai import Task

user_question = "According to the uploaded paper, what is the self-attention mechanism and why does it replace recurrence?"

routing_task = Task(
    description=(
        f"Question: {user_question}\n\n"
        "Decide whether this question should be answered from the uploaded PDF or from a web search. "
        "Output exactly one line in the form: ROUTE=<pdf|web> REASON=<short justification>"
    ),
    expected_output="A single line: ROUTE=<pdf|web> REASON=<short justification>",
    agent=router_agent,
)

retrieval_task = Task(
    description=(
        f"Original question: {user_question}\n\n"
        "Read the router's routing decision from the previous task's output. "
        "Use the matching tool (PDF search tool if ROUTE=pdf, web search tool if ROUTE=web) "
        "to retrieve supporting content, then answer the question grounded in that content. "
        "State which source (PDF or web) the answer came from."
    ),
    expected_output="A grounded answer to the question, with the source (PDF or web) stated explicitly.",
    agent=retriever_agent,
    context=[routing_task],
)

print("Routing and retrieval tasks defined.")

Routing and retrieval tasks defined.


## Step 5: Assemble the Crew and Run

`verbose=True` prints each agent's reasoning trace, which satisfies the assignment's logging/traceability requirement.

In [13]:
from crewai import Crew, Process

crew = Crew(
    agents=[router_agent, retriever_agent],
    tasks=[routing_task, retrieval_task],
    process=Process.sequential,
    verbose=True,
)

# Jupyter kernels run their own event loop, so the sync kickoff() path isn't usable here.
result = await crew.kickoff_async()
print("\n=== Final Answer ===")
print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.12                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4beedd0f-d2c7-4fab-8f2d-0bf1dc3d20a7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Question: According to the uploaded paper, what is the self-attention mechanism and why does it replace  │
│  recurrence?                                                                                                    │
│                                                                                                                 │
│  Decide whether this question should be answered from the uploaded PDF or from a web search. Output exactly     │
│  one line in the form: ROUTE=<pdf|web> REASON=<short justification>                                             │
│  ID: f8d28343-e416-41b2-b450-f0c773bd2bfe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Question: According to the uploaded paper, what is the self-attention mechanism and why does it replace  │
│  recurrence?                                                                                                    │
│                                                                                                                 │
│  Decide whether this question should be answered from the uploaded PDF or from a web search. Output exactly     │
│  one line in the form: ROUTE=<pdf|web> REASON=<short justification>                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retrieval Specialist                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  According to the uploaded paper:                                                                               │
│                                                                                                                 │
│  The self-attention mechanism is a layer that maps a variable-length sequence of symbol representations (x1,    │
│  ..., xn) to another sequence of equal length (z1, ..., zn), where each xi and zi are vectors in some           │
│  dimension d. Self-attention connects all positions in a sequence with a constant number of sequentially        │
│  executed operations, unlike recurrent layers which require a number of sequential operations proportional to   │
│  the sequence length n. This property makes self-attention layers faster than recurrent layers when the         │
│  sequence length is smaller than the representation dimensionality, which is typical in state-of-the-art        │
│  models for sentence representations.                                                                           │
│                                                                                                                 │
│  Key advantages of self-attention compared to recurrence include:                                               │
│                                                                                                                 │
│  - Maximum path length for dependency learning is O(1) for self-attention, compared to O(n) for recurrent       │
│  layers. Shorter paths between input-output positions make it easier to learn long-range dependencies in        │
│  sequences.                                                                                                     │
│                                                                                                                 │
│  - The amount of computation that can be parallelized is higher in self-attention because it requires O(1)      │
│  sequential operations, versus O(n) for recurrent layers.                                                       │
│                                                                                                                 │
│  - While the computational complexity per layer of self-attention is O(n²·d), it remains efficient for typical  │
│  sequence lengths and representation dimensions.                                                                │
│                                                                                                                 │
│  - Self-attention obviates the need for recurrence and convolutions, though to allow the model to understand    │
│  token order, positional encodings are added to input embeddings.                                               │
│                                                                                                                 │
│  Hence, the paper motivates using self-attention by emphasizing its superior parallelizability, ability to      │
│  capture long-range dependencies more effectively due to shorter maximum path lengths, and computational        │
│  advantages compared to recurrence.                                                                             │
│                                                                                                                 │
│  This explanation and rationale appear primarily in Sec

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Original question: According to the uploaded paper, what is the self-attention mechanism and why does    │
│  it replace recurrence?                                                                                         │
│                                                                                                                 │
│  Read the router's routing decision from the previous task's output. Use the matching tool (PDF search tool if  │
│  ROUTE=pdf, web search tool if ROUTE=web) to retrieve supporting content, then answer the question grounded in  │
│  that content. State which source (PDF or web) the answer came from.                                            │
│  Agent: Retrieval Specialist                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4beedd0f-d2c7-4fab-8f2d-0bf1dc3d20a7                                                                       │
│  Final Output: According to the uploaded paper:                                                                 │
│                                                                                                                 │
│  The self-attention mechanism is a layer that maps a variable-length sequence of symbol representations (x1,    │
│  ..., xn) to another sequence of equal length (z1, ..., zn), where each xi and zi are vectors in some           │
│  dimension d. Self-attention connects all positions in a sequence with a constant number of sequentially        │
│  executed operations, unlike recurrent layers which require a number of sequential operations proportional to   │
│  the sequence length n. This property makes self-attention layers faster than recurrent layers when the         │
│  sequence length is smaller than the representation dimensionality, which is typical in state-of-the-art        │
│  models for sentence representations.                                                                           │
│                                                                                                                 │
│  Key advantages of self-attention compared to recurrence include:                                               │
│                                                                                                                 │
│  - Maximum path length for dependency learning is O(1) for self-attention, compared to O(n) for recurrent       │
│  layers. Shorter paths between input-output positions make it easier to learn long-range dependencies in        │
│  sequences.                                                                                                     │
│                                                                                                                 │
│  - The amount of computation that can be parallelized is higher in self-attention because it requires O(1)      │
│  sequential operations, versus O(n) for recurrent layers.                                                       │
│                                                                                                                 │
│  - While the computational complexity per layer of self-attention is O(n²·d), it remains efficient for typical  │
│  sequence lengths and representation dimensions.                                                                │
│                                                                                                                 │
│  - Self-attention obviates the need for recurrence and convolutions, though to allow the model to understand    │
│  token order, positional encodings are added to input embeddings.                                               │
│                                                                                                                 │
│  Hence, the paper motivates using self-attention by emphasizing its superior parallelizability, ability to      │
│  capture long-range dependencies more effectively due to shorter maximum path lengths, and computational        │
│  advantages compared to recurrence.                                                                             │
│                                                                                                                 │
│  This explanation and rationale appear primarily in Se


=== Final Answer ===
According to the uploaded paper:

The self-attention mechanism is a layer that maps a variable-length sequence of symbol representations (x1, ..., xn) to another sequence of equal length (z1, ..., zn), where each xi and zi are vectors in some dimension d. Self-attention connects all positions in a sequence with a constant number of sequentially executed operations, unlike recurrent layers which require a number of sequential operations proportional to the sequence length n. This property makes self-attention layers faster than recurrent layers when the sequence length is smaller than the representation dimensionality, which is typical in state-of-the-art models for sentence representations.

Key advantages of self-attention compared to recurrence include:

- Maximum path length for dependency learning is O(1) for self-attention, compared to O(n) for recurrent layers. Shorter paths between input-output positions make it easier to learn long-range dependencies in se

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 6: Demo Both Retrieval Paths

Run one question expected to route to the PDF and one expected to route to the web, to demonstrate both tools working end-to-end.

In [15]:
async def ask(question: str):
    routing = Task(
        description=(
            f"Question: {question}\n\n"
            "Decide whether this question should be answered from the uploaded PDF or from a web search. "
            "Output exactly one line in the form: ROUTE=<pdf|web> REASON=<short justification>"
        ),
        expected_output="A single line: ROUTE=<pdf|web> REASON=<short justification>",
        agent=router_agent,
    )
    retrieval = Task(
        description=(
            f"Original question: {question}\n\n"
            "Read the router's routing decision from the previous task's output. "
            "Use the matching tool to retrieve supporting content, then answer grounded in that content. "
            "State which source (PDF or web) the answer came from."
        ),
        expected_output="A grounded answer, with the source (PDF or web) stated explicitly.",
        agent=retriever_agent,
        context=[routing],
    )
    demo_crew = Crew(agents=[router_agent, retriever_agent], tasks=[routing, retrieval], process=Process.sequential, verbose=True)
    return await demo_crew.kickoff_async()

pdf_leaning_answer = await ask("What does the Transformer paper say its multi-head attention mechanism computes?")
print("\n=== PDF-leaning demo answer ===")
print(pdf_leaning_answer)

web_leaning_answer = await ask("What are the latest state-of-the-art transformer-based models released in the past year?")
print("\n=== Web-leaning demo answer ===")
print(web_leaning_answer)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.12                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ee3a311f-f9a9-416f-b1df-909309b5b3a1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Question: What does the Transformer paper say its multi-head attention mechanism computes?               │
│                                                                                                                 │
│  Decide whether this question should be answered from the uploaded PDF or from a web search. Output exactly     │
│  one line in the form: ROUTE=<pdf|web> REASON=<short justification>                                             │
│  ID: 42a6664f-b1a7-4da6-a35b-82f151a460d6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Question: What does the Transformer paper say its multi-head attention mechanism computes?               │
│                                                                                                                 │
│  Decide whether this question should be answered from the uploaded PDF or from a web search. Output exactly     │
│  one line in the form: ROUTE=<pdf|web> REASON=<short justification>                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retrieval Specialist                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the complete relevant content about the latest state-of-the-art transformer-based models released in   │
│  the past year, sourced from multiple recent web sources:                                                       │
│                                                                                                                 │
│  1. From "The Evolution of Transformer Architecture: From 2017 to 2024" on Medium:                              │
│  - The 2024 Transformer block represents the current state-of-the-art and includes improvements addressing      │
│  training stability, computational efficiency, and fine-tuning.                                                 │
│  - Key changes include pre-normalization with RMSNorm among other advances.                                     │
│  - These modern Transformer architectures are designed to be faster, more efficient, and more powerful for NLP  │
│  and multi-modal applications.                                                                                  │
│  (Source:                                                                                                       │
│  https://medium.com/@arghya05/the-evolution-of-transformer-architecture-from-2017-to-2024-5a967488e63b)         │
│                                                                                                                 │
│  2. From "Transformer Models Overview" at Emergent Mind:                                                        │
│  - Recent 2023-2024 transformer-based models include:                                                           │
│    - Block-wise Bit-Compression of Transformer-based Models (2023)                                              │
│    - SPT: Fine-Tuning Transformer-based Language Models Efficiently with Sparsification (2023)                  │
│    - Transformer-based Named Entity Recognition with Combined Data Representation (2024)                        │
│    - Linguistic Interpretability of Transformer-based Language Models: systematic reviews expected in 2025      │
│    - Transformer-based Models for Long-Form Document Matching (2023)                                            │
│  - Unified pretraining over text, vision, and audio with cross-modal architectures like dual encoders and       │
│  cross-attention are notable trends.                                                                            │
│  (Source: https://www.emergentmind.com/topics/transformer-based-models)                                         │
│                                                                                                                 │
│  3. From "Transformer alternatives in 2024" on Nebius:                                                          │
│  - Notable new models include:                                                                                  │
│    - StripedHyena-7B: a highly optimized hybrid of attention and gated convolutions using Hyena operators.      │
│    - Mamba: Linear-Time Sequence Modeling with Selective State Spaces.                                          │
│    - Vision Mamba: a Mamba-based model for images scaling well with resolution.                                 │
│  - These models excel at extrapolating to longer sequences and represent an evolution beyond classic            │
│  transformers.                                         

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Original question: What are the latest state-of-the-art transformer-based models released in the past    │
│  year?                                                                                                          │
│                                                                                                                 │
│  Read the router's routing decision from the previous task's output. Use the matching tool to retrieve          │
│  supporting content, then answer grounded in that content. State which source (PDF or web) the answer came      │
│  from.                                                                                                          │
│  Agent: Retrieval Specialist                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 43790bdf-89b3-461c-b96e-46378fadc365                                                                       │
│  Final Output: Here is the complete relevant content about the latest state-of-the-art transformer-based        │
│  models released in the past year, sourced from multiple recent web sources:                                    │
│                                                                                                                 │
│  1. From "The Evolution of Transformer Architecture: From 2017 to 2024" on Medium:                              │
│  - The 2024 Transformer block represents the current state-of-the-art and includes improvements addressing      │
│  training stability, computational efficiency, and fine-tuning.                                                 │
│  - Key changes include pre-normalization with RMSNorm among other advances.                                     │
│  - These modern Transformer architectures are designed to be faster, more efficient, and more powerful for NLP  │
│  and multi-modal applications.                                                                                  │
│  (Source:                                                                                                       │
│  https://medium.com/@arghya05/the-evolution-of-transformer-architecture-from-2017-to-2024-5a967488e63b)         │
│                                                                                                                 │
│  2. From "Transformer Models Overview" at Emergent Mind:                                                        │
│  - Recent 2023-2024 transformer-based models include:                                                           │
│    - Block-wise Bit-Compression of Transformer-based Models (2023)                                              │
│    - SPT: Fine-Tuning Transformer-based Language Models Efficiently with Sparsification (2023)                  │
│    - Transformer-based Named Entity Recognition with Combined Data Representation (2024)                        │
│    - Linguistic Interpretability of Transformer-based Language Models: systematic reviews expected in 2025      │
│    - Transformer-based Models for Long-Form Document Matching (2023)                                            │
│  - Unified pretraining over text, vision, and audio with cross-modal architectures like dual encoders and       │
│  cross-attention are notable trends.                                                                            │
│  (Source: https://www.emergentmind.com/topics/transformer-based-models)                                         │
│                                                                                                                 │
│  3. From "Transformer alternatives in 2024" on Nebius:                                                          │
│  - Notable new models include:                                                                                  │
│    - StripedHyena-7B: a highly optimized hybrid of attention and gated convolutions using Hyena operators.      │
│    - Mamba: Linear-Time Sequence Modeling with Selective State Spaces.                                          │
│    - Vision Mamba: a Mamba-based model for images scaling well with resolution.                                 │
│  - These models excel at extrapolating to longer sequences and represent an evolution beyond classic            │
│  transformers.                                        


=== Web-leaning demo answer ===
Here is the complete relevant content about the latest state-of-the-art transformer-based models released in the past year, sourced from multiple recent web sources:

1. From "The Evolution of Transformer Architecture: From 2017 to 2024" on Medium:
- The 2024 Transformer block represents the current state-of-the-art and includes improvements addressing training stability, computational efficiency, and fine-tuning.
- Key changes include pre-normalization with RMSNorm among other advances.
- These modern Transformer architectures are designed to be faster, more efficient, and more powerful for NLP and multi-modal applications.
(Source: https://medium.com/@arghya05/the-evolution-of-transformer-architecture-from-2017-to-2024-5a967488e63b)

2. From "Transformer Models Overview" at Emergent Mind:
- Recent 2023-2024 transformer-based models include:
  - Block-wise Bit-Compression of Transformer-based Models (2023)
  - SPT: Fine-Tuning Transformer-based Languag

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Next Steps for Submission

1. Drop your domain-specific PDF(s) into `pdfs/` and set `TAVILY_API_KEY`.
2. Replace the `TODO` questions in Steps 4 and 6 with real questions about your own PDF and a web-only topic.
3. Run all cells top to bottom and keep the verbose trace output - it satisfies the logging/traceability requirement.
4. Write the required README covering: system architecture, each agent's logic/responsibilities, coordination flow, and challenges/trade-offs.
5. Submit the notebook (with outputs) and the README through the LMS.